<a href="https://colab.research.google.com/github/MaineCalabrezi13/DiagnosticoPlantas/blob/main/Diagnostico_de_Plantas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install streamlit
!pip install pillow
!pip install pyngrok
!pip install ultralytics

In [27]:
%%writefile app.py

import streamlit as st
from ultralytics import YOLO
from PIL import Image
import os

# ============================================
# CONFIGURAÇÕES DA PÁGINA
# ============================================

st.set_page_config(
    page_title="Diagnóstico Inteligente",
    page_icon="🌱",
    layout="centered"
)

# ============================================
# CARREGAR MODELO TREINADO
# ============================================

@st.cache_resource
def carregar_modelo():

    return YOLO("best.pt")


try:

    modelo = carregar_modelo()

except Exception as erro:

    st.error(
        f"Erro ao carregar o modelo: {erro}"
    )

    st.stop()

# ============================================
# DICIONÁRIO DE EXPLICAÇÕES
# ============================================

RECOMENDACOES = {

    "Apple___Apple_scab":{
        "diagnostico":"Sarna da Maçã",
        "explicacao":"Foram identificadas manchas escuras e lesões características da doença.",
        "acao":"Remover folhas infectadas e aplicar fungicida adequado."
    },

    "Apple___Black_rot":{
        "diagnostico":"Podridão Negra",
        "explicacao":"Foram encontrados padrões associados à podridão negra.",
        "acao":"Remover partes contaminadas e monitorar a propagação."
    },

    "Apple___Cedar_apple_rust":{
        "diagnostico":"Ferrugem da Maçã",
        "explicacao":"Foram identificados sinais típicos de ferrugem na folha.",
        "acao":"Aplicar controle fúngico e remover folhas afetadas."
    },

    "Apple___healthy":{
        "diagnostico":"Folha de Maçã Saudável",
        "explicacao":"Não foram encontrados sinais aparentes de doenças.",
        "acao":"Manter monitoramento periódico."
    },

    "Corn___Cercospora_leaf_spot Gray_leaf_spot":{
        "diagnostico":"Mancha Foliar Cinzenta",
        "explicacao":"Foram detectadas manchas foliares características.",
        "acao":"Aplicar manejo adequado e controlar umidade."
    },

    "Corn___Common_rust":{
        "diagnostico":"Ferrugem Comum do Milho",
        "explicacao":"Foram encontrados sinais típicos de ferrugem.",
        "acao":"Monitorar cultura e aplicar fungicida se necessário."
    },

    "Corn___Northern_Leaf_Blight":{
        "diagnostico":"Queima Foliar do Norte",
        "explicacao":"Foram identificadas áreas de necrose nas folhas.",
        "acao":"Aplicar tratamento adequado."
    },

    "Corn___healthy":{
        "diagnostico":"Milho Saudável",
        "explicacao":"Não foram encontrados sinais aparentes de doenças.",
        "acao":"Manter manejo atual."
    },

    "Potato___Early_blight":{
        "diagnostico":"Requeima Inicial da Batata",
        "explicacao":"Foram identificadas manchas escuras e secas nas folhas.",
        "acao":"Aplicar fungicida e remover folhas afetadas."
    },

    "Potato___Late_blight":{
        "diagnostico":"Requeima Tardia da Batata",
        "explicacao":"Foram encontrados sinais avançados da doença.",
        "acao":"Realizar controle imediato."
    },

    "Potato___healthy":{
        "diagnostico":"Batata Saudável",
        "explicacao":"Não foram encontrados sinais aparentes de doenças.",
        "acao":"Continuar monitoramento."
    },

    "Tomato___Bacterial_spot":{
        "diagnostico":"Mancha Bacteriana",
        "explicacao":"Foram identificadas manchas bacterianas nas folhas.",
        "acao":"Aplicar controle fitossanitário."
    },

    "Tomato___Early_blight":{
        "diagnostico":"Requeima Inicial do Tomate",
        "explicacao":"Foram identificados sintomas iniciais da doença.",
        "acao":"Monitorar e aplicar tratamento."
    },

    "Tomato___Late_blight":{
        "diagnostico":"Requeima Tardia do Tomate",
        "explicacao":"Foram encontrados sintomas avançados.",
        "acao":"Aplicar fungicida e remover folhas afetadas."
    },

    "Tomato___Leaf_Mold":{
        "diagnostico":"Mofo da Folha",
        "explicacao":"Foram encontradas alterações características.",
        "acao":"Reduzir umidade e melhorar ventilação."
    },

    "Tomato___Septoria_leaf_spot":{
        "diagnostico":"Mancha de Septoria",
        "explicacao":"Foram detectadas pequenas manchas circulares.",
        "acao":"Aplicar tratamento adequado."
    },

    "Tomato___Spider_mites Two-spotted_spider_mite":{
        "diagnostico":"Ácaro-Aranha",
        "explicacao":"Foram encontrados danos associados a ácaros.",
        "acao":"Aplicar controle biológico ou químico."
    },

    "Tomato___Target_Spot":{
        "diagnostico":"Mancha Alvo",
        "explicacao":"Foram identificados padrões circulares típicos.",
        "acao":"Realizar tratamento fitossanitário."
    },

    "Tomato___Tomato_mosaic_virus":{
        "diagnostico":"Vírus do Mosaico do Tomateiro",
        "explicacao":"Foram encontradas alterações típicas do mosaico.",
        "acao":"Remover plantas contaminadas."
    },

    "Tomato___Tomato_Yellow_Leaf_Curl_Virus":{
        "diagnostico":"Vírus do Enrolamento Amarelo",
        "explicacao":"Foram detectados sinais de enrolamento e amarelamento.",
        "acao":"Controlar vetores e remover plantas infectadas."
    },

    "Tomato___healthy":{
        "diagnostico":"Tomate Saudável",
        "explicacao":"Não foram encontrados sinais aparentes de doenças.",
        "acao":"Continuar monitoramento."
    }

}

# ============================================
# TÍTULO
# ============================================

st.title(
    "🌱 Sistema Inteligente de Diagnóstico de Plantas"
)

st.write(
"""
Faça o upload de uma imagem para que o sistema realize
uma análise automática.
"""
)

st.divider()

# ============================================
# UPLOAD DA IMAGEM
# ============================================

imagem = st.file_uploader(
    "Selecione uma imagem",
    type=[
        "jpg",
        "jpeg",
        "png",
        "jfif"
    ]
)

# ============================================
# PROCESSAMENTO
# ============================================

if imagem:

    img = Image.open(
        imagem
    )

    col1,col2=st.columns(
        2
    )

# ============================================
# COLUNA DA IMAGEM
# ============================================

    with col1:

        st.subheader(
            "Imagem enviada"
        )

        st.image(
            img,
            use_container_width=True
        )

# ============================================
# IA PROCESSANDO
# ============================================

    with st.spinner(
        "Analisando imagem..."
    ):

        caminho_temp="imagem_temp.jpg"

        img.save(
            caminho_temp
        )

        resultados=modelo(
            caminho_temp,
            verbose=False
        )

        os.remove(
            caminho_temp
        )

# ============================================
# EXTRAÇÃO DOS RESULTADOS
# ============================================

        resultado_ia=resultados[0]

        id_classe=resultado_ia.probs.top1

        nome_classe=resultado_ia.names[
            id_classe
        ]

        confianca=float(
            resultado_ia.probs.top1conf
        )*100

# ============================================
# COLUNA RESULTADOS
# ============================================

    with col2:

        st.subheader(
            "Resultado"
        )

        # Regra de negócio:
        # Não mostrar diagnósticos com baixa confiança

        if confianca < 50:

            st.error(
                "Não foi possível identificar a doença com segurança."
            )

            st.write(
                "A confiança da IA ficou abaixo do limite mínimo estabelecido."
            )

            st.warning(
                "Sugestão: envie uma imagem mais nítida, com boa iluminação e foco na folha."
            )

        else:

            if nome_classe in RECOMENDACOES:

                info=RECOMENDACOES[
                    nome_classe
                ]

                st.success(
                    info[
                        "diagnostico"
                    ]
                )

                st.subheader(
                    "Explicação"
                )

                st.write(
                    info[
                        "explicacao"
                    ]
                )

                st.subheader(
                    "Recomendação"
                )

                st.warning(
                    info[
                        "acao"
                    ]
                )

            else:

                st.success(
                    nome_classe
                )

                st.info(
                    "Classe ainda não cadastrada no sistema."
                )

# ============================================
# CONFIANÇA
# ============================================

        st.subheader(
            "Grau de confiança"
        )

        st.progress(
            int(
                confianca
            )
        )

        st.write(
            f"{confianca:.2f}%"
        )

st.divider()

st.caption(
    "Sistema Inteligente de Diagnóstico de Plantas"
)

Overwriting app.py


In [28]:
!streamlit run app.py &>/content/logs.txt &

In [29]:
from pyngrok import ngrok

ngrok.set_auth_token(
"3EYAAU2ivAmVbHruIz4NdFtoxXt_6g1YPH2DSSudLGBQ3tCQM"
)

In [30]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://breeching-mundane-swiftness.ngrok-free.dev" -> "http://localhost:8501"
